# 09 · The Main Model

**Goal.** Build and train the full detection model.

**Why it matters.** This is the heart of the project. It combines **two language encoders** (one for
meaning, one for emotion) with a **capsule network** and an **attention step** that decides how much to
trust the text vs. the timing signal.

**What happens below.** Assemble the model, train it on YouTube data only, and evaluate it on the
completely unseen Reddit + Twitter test set (the realistic cross-platform setup).

**Output:** the trained model + `data/stage3_model/classification_report.json`.

## 0. Environment & dependencies

In [1]:
import os, sys, json, time, math, logging, warnings, random
from pathlib import Path
from collections import Counter, defaultdict

for _stream in ("stdout", "stderr"):
    try:
        getattr(sys, _stream).reconfigure(encoding="utf-8")
    except Exception:
        pass

import numpy as np

def _try(name):
    try:
        __import__(name); return True
    except Exception:
        return False

assert _try("torch") and _try("transformers") and _try("sklearn"), "notebook 09 needs torch+transformers+sklearn"
HAS_SCIPY = _try("scipy")

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("arch")
PART_VERSION = "1.0"
SEED = 42
random.seed(SEED); np.random.seed(SEED)

hinglish-mental-health-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration & hyperparameters

In [2]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(SEED)

def find_project_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data" / "stage2_semantic").exists() or (d / "data" / "stage1_temporal").exists() or (d / ".env").exists():
            return d
    return here

PROJECT_ROOT = find_project_root()
SEM_DIR  = PROJECT_ROOT / "data" / "stage2_semantic"
TEMP_DIR = PROJECT_ROOT / "data" / "stage1_temporal"
OUTPUT_DIR = PROJECT_ROOT / "data" / "stage3_model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TIER_NAMES = {0: "Neutral", 1: "Stress", 2: "Moderate", 3: "Crisis"}

# ---- backbones (frozen, D2) ----
EMO_CANDIDATES = ["MilaNLProc/xlm-emo-t", "cardiffnlp/twitter-xlm-roberta-base-sentiment"]
EMO_MAX_TOKENS = 128
EMO_BATCH = 16

# ---- model ----
FUSION_DIM   = 128       # common projection dim d (query/key/value)
PRIM_CAPS    = 32        # primary capsules (paper's 32)
PRIM_DIM     = 16        # primary capsule dim (paper's 16)
CLASS_DIM    = 16        # class-capsule dim
N_CLASSES    = 4
ROUTING      = "static"  #  default (dynamic = notebook 10 ablation)
ROUTING_ITERS= 3         # used only if ROUTING="dynamic"
GATE         = "elu"     #  default (none/sigmoid/elu)
FUSION       = "attention"  #  default (attention/concat/text_only)
DROPOUT      = 0.3

# ---- training (head only; backbones frozen) ----
EPOCHS = 60
LR = 1e-3
WEIGHT_DECAY = 1e-4
BATCH = 16
FOCAL_GAMMA = 2.0
CONTRAST_W = 0.2         # supervised-contrastive weight
CONTRAST_T = 0.1         # temperature

log.info("device=%s | routing=%s gate=%s fusion=%s | caps=%dx%d | epochs=%d lr=%.0e",
         DEVICE, ROUTING, GATE, FUSION, PRIM_CAPS, PRIM_DIM, EPOCHS, LR)

2026-08-08 07:51:23,027 | INFO | device=cpu | routing=static gate=elu fusion=attention | caps=32x16 | epochs=60 lr=1e-03


## 2. Load corpus + frozen MuRIL embeddings + temporal features (P3 split asserted)

Rows are aligned by `record_id`: MuRIL mean-pool from notebook 06 (frozen), the six `*_t` temporal features +
`temporal_mask` from notebook 04. TRAIN = YouTube+synthetic; TEST = frozen Reddit+Twitter (leakage guard).

In [3]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(SEM_DIR / "corpus_preprocessed_train.jsonl")
test_rows  = load_jsonl(SEM_DIR / "corpus_preprocessed_test.jsonl")
assert set(r["platform"] for r in train_rows) <= {"youtube", "synthetic"}, "TRAIN platform leak!"
assert set(r["platform"] for r in test_rows)  <= {"reddit", "twitter"},   "TEST platform leak!"

TEMP_FEATS = ["month_t", "day_t", "hour_t", "weekday_t", "is_working_hour_t", "is_weekend_t"]

def load_temporal(split):
    rows = load_jsonl(TEMP_DIR / f"temporal_features_{split}.jsonl")
    by_id = {r["record_id"]: r for r in rows}
    return by_id

temp_train = load_temporal("train")
temp_test  = load_temporal("test")

def muril_matrix(split, rows):
    emb = np.load(SEM_DIR / f"embeddings_{split}.npy").astype(np.float32)
    idx = json.loads((SEM_DIR / f"embeddings_index_{split}.json").read_text(encoding="utf-8"))
    id2row = {rid: i for i, rid in enumerate(idx)}
    return np.vstack([emb[id2row[r["record_id"]]] for r in rows])

MURIL_TR = muril_matrix("train", train_rows)
MURIL_TE = muril_matrix("test",  test_rows)

def temporal_matrix(rows, by_id):
    feats, mask = [], []
    for r in rows:
        t = by_id.get(r["record_id"], {})
        feats.append([float(t.get(k, 0.0)) for k in TEMP_FEATS])
        mask.append(float(t.get("temporal_mask", r.get("temporal_mask", 1))))
    return np.array(feats, np.float32), np.array(mask, np.float32)

TEMP_TR, MASK_TR = temporal_matrix(train_rows, temp_train)
TEMP_TE, MASK_TE = temporal_matrix(test_rows,  temp_test)

y_train = np.array([int(r["tier"]) for r in train_rows])
y_test  = np.array([int(r["tier"]) for r in test_rows])
log.info("train=%d %s | test=%d %s | MuRIL=%s | temporal mask=0 (synthetic) train=%d",
         len(train_rows), {int(t): int((y_train==t).sum()) for t in range(4)},
         len(test_rows),  {int(t): int((y_test==t).sum()) for t in range(4)},
         MURIL_TR.shape, int((MASK_TR == 0).sum()))

2026-08-08 07:51:23,339 | INFO | train=1971 {0: 1545, 1: 82, 2: 255, 3: 89} | test=2768 {0: 1581, 1: 545, 2: 596, 3: 46} | MuRIL=(1971, 768) | temporal mask=0 (synthetic) train=125


## 3. Second backbone — frozen metaphor/emotion embeddings (cached)

A multilingual emotion encoder, frozen. We embed `text_muril` once (mean-pool) and **cache** to
`emo_embeddings_{train,test}.npy` so reruns are instant. If the primary model is unreachable we fall back
through the candidate list; the chosen model id is recorded in the config.

In [4]:
from transformers import AutoTokenizer, AutoModel

def texts_of(rows):
    return [str(r.get("text_muril") or r.get("post_text_clean") or "") for r in rows]

EMO_TR_PATH = OUTPUT_DIR / "emo_embeddings_train.npy"
EMO_TE_PATH = OUTPUT_DIR / "emo_embeddings_test.npy"
EMO_META_PATH = OUTPUT_DIR / "emo_meta.json"

@torch.no_grad()
def emo_embed(model, tok, texts):
    out = []
    for i in range(0, len(texts), EMO_BATCH):
        chunk = [t if t.strip() else "[UNK]" for t in texts[i:i+EMO_BATCH]]
        enc = tok(chunk, padding=True, truncation=True, max_length=EMO_MAX_TOKENS, return_tensors="pt").to(DEVICE)
        hs = model(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).float()
        v = (hs * m).sum(1) / m.sum(1).clamp(min=1e-9)
        out.append(v.cpu().numpy())
    return np.vstack(out).astype(np.float32)

EMO_MODEL_ID = None
if EMO_TR_PATH.exists() and EMO_TE_PATH.exists() and EMO_META_PATH.exists():
    EMO_TR = np.load(EMO_TR_PATH); EMO_TE = np.load(EMO_TE_PATH)
    EMO_MODEL_ID = json.loads(EMO_META_PATH.read_text(encoding="utf-8")).get("model_id")
    log.info("loaded cached emotion embeddings (%s) %s/%s", EMO_MODEL_ID, EMO_TR.shape, EMO_TE.shape)
else:
    for cand in EMO_CANDIDATES:
        try:
            log.info("loading emotion backbone %s (frozen) ...", cand)
            etok = AutoTokenizer.from_pretrained(cand)
            emodel = AutoModel.from_pretrained(cand).to(DEVICE).eval()
            EMO_TR = emo_embed(emodel, etok, texts_of(train_rows))
            EMO_TE = emo_embed(emodel, etok, texts_of(test_rows))
            EMO_MODEL_ID = cand
            np.save(EMO_TR_PATH, EMO_TR); np.save(EMO_TE_PATH, EMO_TE)
            EMO_META_PATH.write_text(json.dumps({"model_id": cand, "dim": int(EMO_TR.shape[1])}), encoding="utf-8")
            del emodel
            break
        except Exception as e:
            log.warning("emotion backbone %s failed: %s", cand, str(e)[:120])
    assert EMO_MODEL_ID is not None, "no emotion backbone could be loaded"
EMO_DIM = EMO_TR.shape[1]
log.info("emotion backbone = %s | dim=%d", EMO_MODEL_ID, EMO_DIM)

2026-08-08 07:51:27,528 | INFO | loaded cached emotion embeddings (MilaNLProc/xlm-emo-t) (1971, 768)/(2768, 768)


2026-08-08 07:51:27,531 | INFO | emotion backbone = MilaNLProc/xlm-emo-t | dim=768


## 4. The model — dual-backbone Capsule + ELU gate + routing + cross-modal attention fusion

**Algorithm 3 (forward pass).** text `x_t` (768, frozen MuRIL), emotion `x_e` (frozen), temporal `x_τ` (6) + mask `m`:
```
H   = Dropout(ELU(W_t x_t))                                  # text projection
G   = gate(H)        (none:1 | sigmoid:σ(W_g H) | elu:ELU(W_g H)+1)        # ELU gate, 
H_g = H ⊙ G
U   = squash(reshape(W_p H_g, [32,16]))                      # Primary Capsules, 
V   = route(U)   (static: coupling=softmax(b); dynamic: 3× agreement)  -> [4,16] class capsules
q   = W_q · flatten(V)                                       # text/capsule query (d)
e   = W_e x_e                                                # emotion key/value (d)
τ   = (W_τ x_τ) ⊙ m                                          # temporal key/value, MASKED -> 0 if synthetic
ctx = softmax(q·[τ;e]ᵀ/√d) · [τ;e]                           # cross-modal attention, 
z   = LayerNorm(q + ctx)        (attention) | [q;τ;e] (concat) | q (text_only)
logits = W_o z ;  z_norm = normalize(z)   (for supervised-contrastive)
```

In [5]:
def squash(s, dim=-1, eps=1e-8):
    n2 = (s * s).sum(dim=dim, keepdim=True)
    return (n2 / (1.0 + n2)) * (s / torch.sqrt(n2 + eps))

class CapsuleRouting(nn.Module):
    def __init__(self, n_in, d_in, n_out, d_out, routing="static", iters=3):
        super().__init__()
        self.n_in, self.n_out, self.routing, self.iters = n_in, n_out, routing, iters
        self.W = nn.Parameter(0.01 * torch.randn(1, n_in, n_out, d_out, d_in))   # transform u_i -> u_hat_ij
        self.b_static = nn.Parameter(torch.zeros(1, n_in, n_out))                # static (input-independent) logits

    def forward(self, u):                       # u: (B, n_in, d_in)
        B = u.size(0)
        u_hat = torch.matmul(self.W, u.unsqueeze(2).unsqueeze(-1)).squeeze(-1)   # (B, n_in, n_out, d_out)
        if self.routing == "static":
            c = torch.softmax(self.b_static, dim=2).unsqueeze(-1)                # (1,n_in,n_out,1)
            s = (c * u_hat).sum(dim=1)                                           # (B, n_out, d_out)
            return squash(s)
        # dynamic routing-by-agreement (Sabour et al. 2017)
        b = torch.zeros(B, self.n_in, self.n_out, device=u.device)
        for it in range(self.iters):
            c = torch.softmax(b, dim=2).unsqueeze(-1)
            s = (c * u_hat).sum(dim=1)
            v = squash(s)                                                        # (B, n_out, d_out)
            if it < self.iters - 1:
                b = b + (u_hat * v.unsqueeze(1)).sum(-1)
        return v

class DriftCapsNet(nn.Module):
    def __init__(self, text_dim, emo_dim, temp_dim=6, d=FUSION_DIM, gate=GATE, fusion=FUSION,
                 routing=ROUTING, dropout=DROPOUT):
        super().__init__()
        self.gate_kind, self.fusion = gate, fusion
        self.text_proj = nn.Linear(text_dim, PRIM_CAPS * PRIM_DIM)
        self.gate_lin  = nn.Linear(PRIM_CAPS * PRIM_DIM, PRIM_CAPS * PRIM_DIM)
        self.caps = CapsuleRouting(PRIM_CAPS, PRIM_DIM, N_CLASSES, CLASS_DIM, routing=routing)
        self.q_proj = nn.Linear(N_CLASSES * CLASS_DIM, d)
        self.e_proj = nn.Linear(emo_dim, d)
        self.t_proj = nn.Sequential(nn.Linear(temp_dim, 64), nn.ReLU(), nn.Linear(64, d))
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d)
        head_in = d if fusion in ("attention", "text_only") else 3 * d
        self.head = nn.Linear(head_in, N_CLASSES)
        self.d = d

    def gate(self, H):
        if self.gate_kind == "none":   return H
        if self.gate_kind == "sigmoid":return H * torch.sigmoid(self.gate_lin(H))
        return H * (F.elu(self.gate_lin(H)) + 1.0)             # elu gate

    def temporal_rep(self, x_t, mask):                         # MASK-AWARE: synthetic -> 0
        return self.t_proj(x_t) * mask.unsqueeze(-1)

    def forward(self, x_text, x_emo, x_temp, mask, return_z=False):
        H = self.drop(F.elu(self.text_proj(x_text)))
        H_g = self.gate(H)
        U = squash(H_g.view(-1, PRIM_CAPS, PRIM_DIM))
        V = self.caps(U).reshape(x_text.size(0), -1)           # (B, 4*16)
        q = self.q_proj(V)                                     # (B, d) text/capsule query
        e = self.e_proj(x_emo)                                 # (B, d)
        tau = self.temporal_rep(x_temp, mask)                  # (B, d), masked
        if self.fusion == "attention":
            kv = torch.stack([tau, e], dim=1)                  # (B, 2, d)
            att = torch.softmax(torch.matmul(q.unsqueeze(1), kv.transpose(1, 2)) / math.sqrt(self.d), dim=-1)
            ctx = torch.matmul(att, kv).squeeze(1)             # (B, d)
            z = self.norm(q + ctx)
        elif self.fusion == "concat":
            z = torch.cat([q, tau, e], dim=-1)
        else:                                                  # text_only
            z = q
        logits = self.head(z)
        return (logits, F.normalize(z, dim=-1)) if return_z else logits

log.info("model defined (DriftCapsNet)")

2026-08-08 07:51:27,562 | INFO | model defined (DriftCapsNet)


## 5. Losses — focal (class-weighted) + supervised contrastive

In [6]:
class FocalLoss(nn.Module):
    def __init__(self, weight, gamma=FOCAL_GAMMA):
        super().__init__()
        self.weight = weight; self.gamma = gamma
    def forward(self, logits, y):
        logp = F.log_softmax(logits, dim=-1)
        p = logp.exp()
        w = self.weight[y]
        pt = p.gather(1, y.unsqueeze(1)).squeeze(1).clamp(1e-6, 1.0)
        logpt = logp.gather(1, y.unsqueeze(1)).squeeze(1)
        return (-w * (1 - pt) ** self.gamma * logpt).mean()

def supervised_contrastive(z, y, temp=CONTRAST_T):
    # Khosla et al. 2020 — pull same-Tier embeddings together within the batch
    sim = torch.matmul(z, z.t()) / temp                        # (B,B)
    B = z.size(0)
    logits_mask = ~torch.eye(B, dtype=torch.bool, device=z.device)
    pos = (y.unsqueeze(0) == y.unsqueeze(1)) & logits_mask
    sim = sim.masked_fill(~logits_mask, -1e9)
    logp = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    pos_cnt = pos.sum(1)
    valid = pos_cnt > 0
    if valid.sum() == 0:
        return torch.tensor(0.0, device=z.device)
    mean_pos = (logp.masked_fill(~pos, 0.0).sum(1)[valid]) / pos_cnt[valid].clamp(min=1)
    return -mean_pos.mean()

log.info("losses ready (focal gamma=%.1f + supcon w=%.2f)", FOCAL_GAMMA, CONTRAST_W)

2026-08-08 07:51:27,583 | INFO | losses ready (focal gamma=2.0 + supcon w=0.20)


## 6. Train (head only; backbones frozen) on TRAIN, ready to evaluate on frozen TEST

In [7]:
from torch.utils.data import DataLoader, TensorDataset

def to_t(*arrs):
    return [torch.tensor(a) for a in arrs]

Xt_tr, Xe_tr, Xtau_tr, M_tr, Y_tr = to_t(MURIL_TR, EMO_TR, TEMP_TR, MASK_TR, y_train)
Xt_te, Xe_te, Xtau_te, M_te, Y_te = to_t(MURIL_TE, EMO_TE, TEMP_TE, MASK_TE, y_test)
ds = TensorDataset(Xt_tr, Xe_tr, Xtau_tr, M_tr, Y_tr.long())
g = torch.Generator(); g.manual_seed(SEED)
dl = DataLoader(ds, batch_size=BATCH, shuffle=True, generator=g)

counts = np.bincount(y_train, minlength=4).astype(float)
cls_w = torch.tensor(counts.sum() / (4.0 * np.clip(counts, 1, None)), dtype=torch.float, device=DEVICE)

torch.manual_seed(SEED)
model = DriftCapsNet(MURIL_TR.shape[1], EMO_DIM).to(DEVICE)
focal = FocalLoss(cls_w)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
log.info("trainable params (head; backbones frozen) = %d", n_params)

t0 = time.time()
model.train()
for ep in range(EPOCHS):
    tot = 0.0
    for xt, xe, xtau, m, yb in dl:
        xt, xe, xtau, m, yb = xt.to(DEVICE), xe.to(DEVICE), xtau.to(DEVICE), m.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        logits, z = model(xt, xe, xtau, m, return_z=True)
        loss = focal(logits, yb) + CONTRAST_W * supervised_contrastive(z, yb)
        loss.backward(); opt.step(); tot += float(loss)
    if (ep + 1) % 10 == 0 or ep == 0:
        log.info("  epoch %2d/%d loss=%.4f", ep + 1, EPOCHS, tot / max(1, len(dl)))
train_secs = round(time.time() - t0, 1)
log.info("trained in %.1fs", train_secs)

2026-08-08 07:51:27,679 | INFO | trainable params (head; backbones frozen) = 805572


2026-08-08 07:51:30,143 | INFO |   epoch  1/60 loss=0.9316


2026-08-08 07:51:55,314 | INFO |   epoch 10/60 loss=0.5152


2026-08-08 07:52:12,434 | INFO |   epoch 20/60 loss=0.4751


2026-08-08 07:52:28,157 | INFO |   epoch 30/60 loss=0.4555


2026-08-08 07:52:52,651 | INFO |   epoch 40/60 loss=0.4618


2026-08-08 07:53:22,786 | INFO |   epoch 50/60 loss=0.4572


2026-08-08 07:54:02,672 | INFO |   epoch 60/60 loss=0.4440


2026-08-08 07:54:02,675 | INFO | trained in 155.0s


## 7. Assert the temporal mask zeroes the synthetic temporal contribution (carry-in invariant 2)

In [8]:
model.eval()
with torch.no_grad():
    synth_idx = np.where(MASK_TR == 0)[0]
    if len(synth_idx):
        i = int(synth_idx[0])
        tr_masked = model.temporal_rep(Xt_tau0 := torch.tensor(TEMP_TR[i:i+1]).to(DEVICE),
                                       torch.tensor(MASK_TR[i:i+1]).to(DEVICE))
        real_idx = int(np.where(MASK_TR == 1)[0][0])
        tr_real = model.temporal_rep(torch.tensor(TEMP_TR[real_idx:real_idx+1]).to(DEVICE),
                                     torch.tensor(MASK_TR[real_idx:real_idx+1]).to(DEVICE))
        synth_zero = bool(torch.allclose(tr_masked, torch.zeros_like(tr_masked)))
        real_nonzero = bool(tr_real.abs().sum() > 0)
        log.info("temporal_mask assertion: synthetic temporal rep == 0 -> %s ; real row nonzero -> %s",
                 synth_zero, real_nonzero)
        assert synth_zero, "temporal_mask did NOT zero a synthetic row's temporal contribution!"
    else:
        synth_zero, real_nonzero = True, True
        log.info("no synthetic (mask=0) rows present to assert on")

2026-08-08 07:54:02,697 | INFO | temporal_mask assertion: synthetic temporal rep == 0 -> True ; real row nonzero -> True


## 8. Evaluate on the frozen TEST — full classification report (P/R/F1, confusion, MCC, ROC-AUC)

In [9]:
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             confusion_matrix, matthews_corrcoef, roc_auc_score)

@torch.no_grad()
def predict(Xt, Xe, Xtau, M):
    model.eval()
    logits = model(Xt.to(DEVICE), Xe.to(DEVICE), Xtau.to(DEVICE), M.to(DEVICE))
    prob = torch.softmax(logits, dim=-1).cpu().numpy()
    return logits.argmax(-1).cpu().numpy(), prob

pred, prob = predict(Xt_te, Xe_te, Xtau_te, M_te)
P, R, Fa, _ = precision_recall_fscore_support(y_test, pred, labels=[0,1,2,3], zero_division=0)
cm = confusion_matrix(y_test, pred, labels=[0,1,2,3]).tolist()
try:
    auc = float(roc_auc_score(y_test, prob, multi_class="ovr", average="macro", labels=[0,1,2,3]))
except Exception as e:
    auc = None; log.warning("ROC-AUC failed: %s", e)

report = {
    "part": "stage3_part3.2_architecture", "version": PART_VERSION,
    "principle": "P1; P2 (text primary, temporal auxiliary); P3 (train=youtube+synthetic, test=reddit+twitter FROZEN); "
                 " dual backbone;  Tier-A temporal;  routing;  ELU gate;  cross-modal fusion;  focal+contrastive",
    "config": {"routing": ROUTING, "gate": GATE, "fusion": FUSION, "prim_caps": PRIM_CAPS, "prim_dim": PRIM_DIM,
               "class_dim": CLASS_DIM, "fusion_dim": FUSION_DIM, "epochs": EPOCHS, "lr": LR, "batch": BATCH,
               "focal_gamma": FOCAL_GAMMA, "contrast_w": CONTRAST_W, "frozen_backbones": True, "seed": SEED},
    "backbones": {"text": "google/muril-base-cased (mean-pool, frozen)", "emotion": EMO_MODEL_ID + " (frozen)"},
    "trainable_params": int(n_params), "train_seconds": train_secs, "device": DEVICE,
    "test_metrics": {
        "accuracy": round(float(accuracy_score(y_test, pred)), 4),
        "macro_f1": round(float(f1_score(y_test, pred, average="macro", zero_division=0)), 4),
        "mcc": round(float(matthews_corrcoef(y_test, pred)), 4),
        "roc_auc_ovr_macro": round(auc, 4) if auc is not None else None,
        "per_tier": {TIER_NAMES[i]: {"P": round(float(P[i]),4), "R": round(float(R[i]),4), "F1": round(float(Fa[i]),4)} for i in range(4)},
        "tier3_recall": round(float(R[3]), 4),
        "confusion_matrix": cm,
    },
    "temporal_mask_assert": {"synthetic_temporal_zero": bool(synth_zero), "real_temporal_nonzero": bool(real_nonzero)},
    "honest_note": "Pilot-scale architecture-correctness proof (202 TRAIN rows): real end-to-end forward/backward, "
                   "real frozen-TEST predictions, all modules wired (dual backbone + capsule + routing + ELU gate + "
                   "cross-modal attention fusion + focal+contrastive). Backbones FROZEN per the notebook 07 finding "
                   "that fine-tuning 202 rows hurts. Numbers are provisional; the publishable model = data scale-up + "
                   "GPU + Optuna (, notebook 10). P-series honesty.",
}
print("=== FROZEN-TEST METRICS ===")
print(f"acc={report['test_metrics']['accuracy']}  macroF1={report['test_metrics']['macro_f1']}  "
      f"MCC={report['test_metrics']['mcc']}  ROC-AUC={report['test_metrics']['roc_auc_ovr_macro']}  "
      f"T3rec={report['test_metrics']['tier3_recall']}")
for t in range(4):
    pt = report["test_metrics"]["per_tier"][TIER_NAMES[t]]
    print(f"  {TIER_NAMES[t]:9} P={pt['P']:.3f} R={pt['R']:.3f} F1={pt['F1']:.3f}")
print("confusion (rows=true 0..3):", cm)

=== FROZEN-TEST METRICS ===
acc=0.7023  macroF1=0.4916  MCC=0.4572  ROC-AUC=0.8355  T3rec=0.1087
  Neutral   P=0.697 R=0.943 F1=0.802
  Stress    P=0.863 R=0.380 F1=0.527
  Moderate  P=0.668 R=0.404 F1=0.504
  Crisis    P=0.172 R=0.109 F1=0.133
confusion (rows=true 0..3): [[1491, 20, 63, 7], [294, 207, 40, 4], [333, 9, 241, 13], [20, 4, 17, 5]]


## 9. Persist model + predictions + report

In [10]:
torch.save(model.state_dict(), OUTPUT_DIR / "model_state.pt")
(OUTPUT_DIR / "model_config.json").write_text(json.dumps(report["config"] | {
    "text_dim": int(MURIL_TR.shape[1]), "emo_dim": int(EMO_DIM), "emo_model": EMO_MODEL_ID,
    "temp_feats": TEMP_FEATS}, indent=2), encoding="utf-8")

with open(OUTPUT_DIR / "predictions_test.jsonl", "w", encoding="utf-8") as fh:
    for r, p, pr in zip(test_rows, pred, prob):
        fh.write(json.dumps({"record_id": r["record_id"], "tier_true": int(r["tier"]),
                             "tier_pred": int(p), "probs": [round(float(x), 4) for x in pr]},
                            ensure_ascii=False) + "\n")

(OUTPUT_DIR / "classification_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
log.info("wrote model_state.pt, model_config.json, predictions_test.jsonl, classification_report.json")

2026-08-08 07:54:03,143 | INFO | wrote model_state.pt, model_config.json, predictions_test.jsonl, classification_report.json


## 10. Architecture card & This step is complete when

**notebook 09 is complete when:** the model **trains end-to-end** (real forward/backward) on TRAIN and **predicts
the frozen TEST**; `classification_report.json` has per-Tier P/R/F1 + 4×4 confusion + MCC + ROC-AUC; the
**`temporal_mask` is asserted to zero synthetic temporal contribution** (carry-in invariant 2); the dual
backbone, Capsule+routing, ELU gate, cross-modal attention fusion, and
focal+contrastive loss are all wired and exercised; backbones frozen. Honest pilot-scale caveat
written. **No human; P3 leakage guard asserted.**

In [11]:
print("=" * 64); print("Objective 3 · notebook 09 — ARCHITECTURE CARD"); print("=" * 64)
print("backbones      : MuRIL mean-pool (frozen) +", EMO_MODEL_ID, "(frozen)")
print("modules        : ELU-gate -> PrimaryCaps(32x16) ->", ROUTING, "routing -> 4 class caps ->", FUSION, "fusion -> head")
print("loss           : focal(gamma=%.1f, class-weighted) + %.2f*supcon" % (FOCAL_GAMMA, CONTRAST_W))
print("trainable params:", n_params, "| train secs:", train_secs)
m = report["test_metrics"]
print("frozen TEST    : acc=%.3f macroF1=%.3f MCC=%.3f ROC-AUC=%s T3rec=%.3f" % (
    m["accuracy"], m["macro_f1"], m["mcc"], m["roc_auc_ovr_macro"], m["tier3_recall"]))
print("temporal_mask  : synthetic temporal rep == 0 ->", report["temporal_mask_assert"]["synthetic_temporal_zero"])
dod = all([
    (OUTPUT_DIR / "classification_report.json").exists(),
    (OUTPUT_DIR / "model_state.pt").exists(),
    (OUTPUT_DIR / "predictions_test.jsonl").exists(),
    report["temporal_mask_assert"]["synthetic_temporal_zero"],
])
print("\nCompletion check:", "PASS" if dod else "INCOMPLETE", "(pilot-scale architecture-correctness proof)")

Objective 3 · notebook 09 — ARCHITECTURE CARD
backbones      : MuRIL mean-pool (frozen) + MilaNLProc/xlm-emo-t (frozen)
modules        : ELU-gate -> PrimaryCaps(32x16) -> static routing -> 4 class caps -> attention fusion -> head
loss           : focal(gamma=2.0, class-weighted) + 0.20*supcon
trainable params: 805572 | train secs: 155.0
frozen TEST    : acc=0.702 macroF1=0.492 MCC=0.457 ROC-AUC=0.8355 T3rec=0.109
temporal_mask  : synthetic temporal rep == 0 -> True

Completion check: PASS (pilot-scale architecture-correctness proof)
